# 03 — Modeling + SHAP

Мета: навчити LightGBM модель, оптимізувати поріг для F1, оцінити метрики та пояснити результати через SHAP.

**Цільова метрика:** F1-score для класу fraud > 0.6

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
from src.model import train_model, predict_scores

## 1. Завантаження processed фічей

In [ ]:
train_df = pd.read_csv('../../data/processed/train_features.csv')
test_df = pd.read_csv('../../data/processed/test_features.csv')

print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')
train_df.head()

## 2. Навчання LightGBM
Модель з `scale_pos_weight` для боротьби з дисбалансом та оптимізацією порогу для F1.

In [ ]:
model, metrics, feature_cols, threshold, importance = train_model(train_df)

print(f"\n{'='*50}")
print(f"📊 ROC-AUC:        {metrics['roc_auc']:.4f}")
print(f"🎯 Best F1 (fraud): {metrics['best_f1']:.4f}")
print(f"📏 Threshold:       {metrics['best_threshold']:.2f}")
print(f"{'='*50}")
print(f"\n{metrics['classification_report']}")

## 3. Feature Importance

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

top_n = 15
top_features = importance.head(top_n)

plt.figure(figsize=(10, 8))
sns.barplot(x=top_features.values, y=top_features.index, orient='h', palette='viridis')
plt.title(f'Top-{top_n} Feature Importance (LightGBM)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 4. SHAP Analysis

In [ ]:
import shap

X_train = train_df.drop(columns=['is_fraud', 'id_user'], errors='ignore')

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train.sample(min(1000, len(X_train)), random_state=42))

# Якщо shap_values — list (binary classification), беремо клас 1
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

shap.summary_plot(sv, X_train.sample(min(1000, len(X_train)), random_state=42), max_display=15)

## 5. Прогнози на тестовому наборі

In [ ]:
import os
os.makedirs('../../outputs', exist_ok=True)

predictions = predict_scores(model, test_df, feature_cols, threshold)
predictions.to_csv('../../outputs/predictions.csv', index=False)

print(f"✅ Predictions saved to outputs/predictions.csv")
print(f"   Total test users: {len(predictions):,}")
print(f"   Predicted fraud:  {predictions['is_fraud'].sum():,} ({predictions['is_fraud'].mean()*100:.2f}%)")
predictions.head(10)